# 02. Очистка данных и подготовка меток

## Цель

Проверить пригодность последовательностей и аннотаций, согласовать правила формирования классов, обработать неоднозначные записи и точные повторы, подготовить очищенный набор данных для дальнейшей работы.

*В ноутбуке не выполняются обучение моделей, подбор признаков и исследование различий аминокислотного состава между классами.*

## Импорт библиотек

In [1]:
import pandas as pd
import re

import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.bioinformatic_functions import get_locations, get_label, get_class

## Загрузка tsv файла

In [2]:
df_raw = pd.read_csv("../data/raw/uniprot_human_swissprot_raw.tsv", sep="\t")

df = df_raw.copy()

print("Размер dataframe:", df.shape)

df.head()

Размер dataframe: (17342, 8)


,Entry,Entry Name,Protein names,Gene Names,Organism,Length,Sequence,Subcellular location [CC]
0,A0A087X1C5,CP2D7_HUMAN,Cytochrome P450 2D7 (EC 1.14.14.1),CYP2D7,Homo sapiens (Human),515,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...
1,A0A096LP01,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,Homo sapiens (Human),95,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,SUBCELLULAR LOCATION: Mitochondrion outer memb...
2,A0A0B4J2F0,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,Homo sapiens (Human),54,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,SUBCELLULAR LOCATION: Mitochondrion outer memb...
3,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),16,MRWQEMGYIFYPRKLR,SUBCELLULAR LOCATION: Secreted {ECO:0000269|Pu...
4,A0A0K2S4Q6,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,Homo sapiens (Human),201,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...


## Проверка аминокислотного алфавита

В первой версии оставляем последовательности только из 20 стандартных
аминокислот. Записи с другими символами сохраняем отдельно.

Ограничения длины пока не вводим.

In [3]:
mask = df["Sequence"].str.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]+", na=False)

df_other_symbols = df.loc[~mask].copy()

print("Последовательностей вне стандартного алфавита:", len(df_other_symbols))

Последовательностей вне стандартного алфавита: 23


In [4]:
df = df.loc[mask].copy()

print("Осталось записей:", len(df))

Осталось записей: 17319


## Записи с отдельными аннотациями изоформ

Некоторые записи описывают разные локализации для разных изоформ.

В первой версии такие записи откладываем, чтобы не назначить
последовательности локализацию другого варианта белка

In [5]:
mask = df["Subcellular location [CC]"].str.contains(r"\[\s*Isoform", case=False, regex=True, na=False)

df_isoforms = df.loc[mask].copy()

print(f"Записей с аннотациями изоформ:", len(df_isoforms))

df = df.loc[~mask].copy()

print("Осталось записей:", len(df))

Записей с аннотациями изоформ: 762
Осталось записей: 16557


## Выделение локализаций из текста

In [6]:
df["locations"] = df["Subcellular location [CC]"].apply(get_locations)

with pd.option_context("display.max_colwidth", None):
    display(
        df[["Entry", "Subcellular location [CC]", "locations"]].head(10)
    )

,Entry,Subcellular location [CC],locations
0,A0A087X1C5,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; Multi-pass membrane protein {ECO:0000255}. Cytoplasm {ECO:0000305|PubMed:15051713}. Mitochondrion {ECO:0000269|PubMed:18838503}.,"[(Membrane, [ECO:0000305]), (Cytoplasm, [ECO:0000305]), (Mitochondrion, [ECO:0000269])]"
1,A0A096LP01,"SUBCELLULAR LOCATION: Mitochondrion outer membrane {ECO:0000269|PubMed:33386847, ECO:0000269|PubMed:37009826}; Single-pass membrane protein {ECO:0000255}.","[(Mitochondrion outer membrane, [ECO:0000269, ECO:0000269])]"
2,A0A0B4J2F0,SUBCELLULAR LOCATION: Mitochondrion outer membrane {ECO:0000269|PubMed:31653868}; Single-pass membrane protein {ECO:0000255}.,"[(Mitochondrion outer membrane, [ECO:0000269])]"
3,A0A0C5B5G6,"SUBCELLULAR LOCATION: Secreted {ECO:0000269|PubMed:25738459}. Mitochondrion {ECO:0000269|PubMed:29983246}. Nucleus {ECO:0000269|PubMed:29983246, ECO:0000269|PubMed:33473109}. Note=Translocates to the nucleus in response to metabolic stress in an AMPK-dependent manner. {ECO:0000269|PubMed:29983246}.","[(Secreted, [ECO:0000269]), (Mitochondrion, [ECO:0000269]), (Nucleus, [ECO:0000269, ECO:0000269])]"
5,A0A0U1RRE5,"SUBCELLULAR LOCATION: Cytoplasm, P-body {ECO:0000269|PubMed:27918561}. Note=Localizes to P-bodies at low concentrations without dissociating them. {ECO:0000269|PubMed:27918561}.","[(Cytoplasm, P-body, [ECO:0000269])]"
6,A0A1B0GTW7,SUBCELLULAR LOCATION: Membrane {ECO:0000255}; Single-pass type I membrane protein {ECO:0000255}.,"[(Membrane, [ECO:0000255])]"
7,A0A2R8Y7D0,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|PubMed:36369429}. Cell junction {ECO:0000269|PubMed:36369429}. Cytoplasm {ECO:0000250|UniProtKB:A0A1B0GRQ3}.,"[(Nucleus, [ECO:0000269]), (Cell junction, [ECO:0000269]), (Cytoplasm, [ECO:0000250])]"
8,A0A8I5KQE6,"SUBCELLULAR LOCATION: Cell membrane {ECO:0000255|HAMAP-Rule:MF_03016}. Cytoplasm {ECO:0000255|HAMAP-Rule:MF_03016}. Nucleus {ECO:0000255|HAMAP-Rule:MF_03016}. Note=67LR is found at the surface of the plasma membrane, with its C-terminal laminin-binding domain accessible to extracellular ligands. 37LRP is found at the cell surface, in the cytoplasm and in the nucleus. Co-localizes with PPP1R16B in the cell membrane. {ECO:0000255|HAMAP-Rule:MF_03016}.","[(Cell membrane, [ECO:0000255]), (Cytoplasm, [ECO:0000255]), (Nucleus, [ECO:0000255])]"
9,A0AV02,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; Multi-pass membrane protein {ECO:0000305}.,"[(Membrane, [ECO:0000305])]"
10,A0AV96,"SUBCELLULAR LOCATION: Nucleus {ECO:0000269|PubMed:24038582, ECO:0000269|PubMed:24916387}. Cytoplasm {ECO:0000269|PubMed:24916387}.","[(Nucleus, [ECO:0000269, ECO:0000269]), (Cytoplasm, [ECO:0000269])]"


## Соответствие локализаций нашим классам и назначение метки

Используем шесть предварительных классов:

- nucleus;
- cytoplasm;
- cell_membrane;
- mitochondrion;
- secreted;
- endoplasmic_reticulum.

Оставляем запись, если:

1. Все выделенные локализации распознаны.
2. Они относятся к одному нашему классу.
3. Хотя бы одно утверждение об этом классе содержит ECO:0000269.

Если указана неизвестная локализация или несколько разных классов,
запись пока откладываем.

In [7]:
df["label"] = df["locations"].apply(get_label)

print("Записей с предварительной меткой:", df["label"].notna().sum())
print("Записей без метки:", df["label"].isna().sum())

df["label"]

Записей с предварительной меткой: 3327
Записей без метки: 13230


0                  NaN
1        mitochondrion
2        mitochondrion
3                  NaN
5            cytoplasm
             ...      
17337              NaN
17338              NaN
17339              NaN
17340              NaN
17341              NaN
Name: label, Length: 16557, dtype: str

Отсутствие метки не означает ошибку исходной записи. Запись могла быть исключена из-за нескольких локализаций, отсутствия подходящего экспериментального кода или отсутствия термина в словаре.

In [8]:
df["has_note"] = df["Subcellular location [CC]"].str.contains(r"\bNote\s*=", regex=True, na=False)

df_notes_review = df.loc[df["has_note"]].copy()

print("Всего записей с примечаниями:", len(df_notes_review))
print("C предварительной меткой:", df_notes_review["label"].notna().sum())

Всего записей с примечаниями: 5781
C предварительной меткой: 781


## Просмотр нераспознанных терминов

In [9]:
unknown_locations = []

for locations in df["locations"]:
    for location, evidence in locations:
        if get_class(location) is None:
            unknown_locations.append(location)

pd.Series(unknown_locations, dtype="string").value_counts().head(30)

Membrane                                         1494
Chromosome                                        469
Golgi apparatus membrane                          438
Lysosome membrane                                 254
Golgi apparatus                                   233
Apical cell membrane                              226
Cell projection, dendrite                         199
Nucleus speckle                                   191
Cell projection, cilium                           180
Cell projection, axon                             169
Endosome membrane                                 160
Synapse                                           155
Early endosome membrane                           153
Cytoplasmic vesicle membrane                      137
Lysosome                                          136
Late endosome membrane                            135
Basolateral cell membrane                         132
Early endosome                                    130
Cytoplasmic vesicle         

## Проверка одинаковых последовательностей

In [10]:
groups = df.groupby("Sequence").agg(
    records=("Entry", "size"),
    labeled_records=("label", "count"),
    different_labels=("label", "nunique"),
    has_note=("has_note", "any")
)

good_sequences = groups.index[
    (groups["records"] == groups["labeled_records"])
    & (groups["different_labels"] == 1)
    & ~groups["has_note"]
]

isoform_sequences = set(df_isoforms["Sequence"].dropna())

good_sequences = [
    sequence
    for sequence in good_sequences
    if sequence not in isoform_sequences
]

df_clean = (
    df.loc[df["Sequence"].isin(good_sequences)]
    .sort_values("Entry")
    .drop_duplicates(subset="Sequence")
    .copy()
)

print("Итоговое количество последовательностей:", len(df_clean))

df_clean["label"].value_counts()

Итоговое количество последовательностей: 2545


label
nucleus                  735
cytoplasm                520
cell_membrane            420
mitochondrion            404
secreted                 258
endoplasmic_reticulum    208
Name: count, dtype: int64

## Проверка разметки

In [11]:
with pd.option_context("display.max_colwidth", None):
    for label in sorted(df_clean["label"].unique()):
        print("\nКласс:", label)

        display(df_clean.loc[df_clean["label"] == label, ["Entry", "Subcellular location [CC]", "label"]].head(3))


Класс: cell_membrane


,Entry,Subcellular location [CC],label
15416,A1L1A6,SUBCELLULAR LOCATION: Cell membrane {ECO:0000269|PubMed:31560140}; Single-pass membrane protein {ECO:0000255}.,cell_membrane
110,A6NNB3,SUBCELLULAR LOCATION: Cell membrane {ECO:0000269|PubMed:24519609}; Multi-pass membrane protein {ECO:0000255}.,cell_membrane
13712,A6NND4,SUBCELLULAR LOCATION: Cell membrane {ECO:0000269|PubMed:24999593}; Multi-pass membrane protein {ECO:0000255}.,cell_membrane



Класс: cytoplasm


,Entry,Subcellular location [CC],label
15418,A2A3N6,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000269|PubMed:17623810}.,cytoplasm
57,A4D126,"SUBCELLULAR LOCATION: Cytoplasm, cytosol {ECO:0000269|PubMed:26687144}.",cytoplasm
13677,A4QMS7,"SUBCELLULAR LOCATION: Cytoplasm, cytoskeleton, cilium axoneme {ECO:0000269|PubMed:36191189}. Cytoplasm, cytoskeleton, flagellum axoneme {ECO:0000250|UniProtKB:Q9DAR0}.",cytoplasm



Класс: endoplasmic_reticulum


,Entry,Subcellular location [CC],label
16874,A0A494C1I1,SUBCELLULAR LOCATION: Endoplasmic reticulum membrane {ECO:0000269|PubMed:34897904}; Single-pass membrane protein {ECO:0000255}.,endoplasmic_reticulum
12,A0AVI4,SUBCELLULAR LOCATION: Endoplasmic reticulum membrane {ECO:0000269|PubMed:24807418}; Multi-pass membrane protein {ECO:0000269|PubMed:24807418}.,endoplasmic_reticulum
32,A1L0T0,SUBCELLULAR LOCATION: Endoplasmic reticulum membrane {ECO:0000269|PubMed:28289220}; Single-pass membrane protein {ECO:0000255}.,endoplasmic_reticulum


... (2 outputs left out)


Класс: nucleus


,Entry,Subcellular location [CC],label
15393,A0A455ZAR2,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|PubMed:30367041}.,nucleus
13662,A0A7P0TBJ1,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|PubMed:36631436}.,nucleus
13,A0AVK6,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|PubMed:15897886}.,nucleus



Класс: secreted


,Entry,Subcellular location [CC],label
15350,A0A0J9YXV3,SUBCELLULAR LOCATION: Secreted {ECO:0000269|PubMed:33510483}.,secreted
123,A8K2U0,SUBCELLULAR LOCATION: Secreted {ECO:0000269|PubMed:16298998}.,secreted
204,O00175,"SUBCELLULAR LOCATION: Secreted {ECO:0000269|PubMed:9104803, ECO:0000269|PubMed:9365122}.",secreted


## Технические проверки перед сохранением

In [12]:
assert df_clean["Sequence"].notna().all()
assert df_clean["label"].notna().all()

assert df_clean["Sequence"].is_unique
assert df_clean["Entry"].is_unique

assert df_clean["Sequence"].str.len().eq(df_clean["Length"]).all()

assert df_clean["Sequence"].str.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]+", na=False).all()
assert not df_clean["Subcellular location [CC]"].str.contains(r"\bNote\s*=", regex=True, na=False).any()

allowed_labels = {
    "nucleus",
    "cytoplasm",
    "cell_membrane",
    "mitochondrion",
    "secreted",
    "endoplasmic_reticulum"
}

assert set(df_clean["label"]).issubset(allowed_labels)

print("Технические проверки пройдены.")

Технические проверки пройдены.


## Сохранение

Сохраняем предварительно размеченный набор и исходные записи, которые в него не вошли.

Исходный TSV не изменяем. Разбиение на обучение и тест пока не выполняем.

In [13]:
columns = [
    "Entry",
    "Protein names",
    "Length",
    "Sequence",
    "Subcellular location [CC]",
    "label"
]

df_clean[columns].to_csv("../data/interim/proteins_labeled_candidates.tsv", sep="\t", index=False)

## Выводы

Исходная таблица содержала 17 342 записи о белках человека.

На первом этапе отложены 23 записи с символами вне алфавита 20 стандартных аминокислот. Затем отложены 762 записи с изоформ-специфичными аннотациями локализации.

Для оставшихся 16 557 записей выполнено выделение локализаций из текстовых аннотаций. По заданным правилам предварительные метки получили 3 327 записей.

Отсутствие метки не означает ошибку исходной записи. Причиной могли быть несколько локализаций, отсутствие подходящего экспериментального кода или отсутствие термина в используемом словаре.

Для первой версии набора временно отложены группы последовательностей, в которых хотя бы одна запись содержала примечание Note=. Такие примечания могут описывать дополнительные или условные локализации.

Одинаковые последовательности проверялись группами. Сохранялись только группы, в которых все рассматриваемые записи получили одинаковую метку.После этого оставлена одна строка на последовательность.

Итоговый предварительно размеченный набор содержит 2 545 уникальных последовательностей.

**Распределение классов**

| Класс | Число последовательностей |
|---|---:|
| Ядро | 735 |
| Цитоплазма | 520 |
| Плазматическая мембрана | 420 |
| Митохондрия | 404 |
| Секретируемые белки | 258 |
| Эндоплазматический ретикулум | 208 |
| Всего | 2 545 |

Классы представлены неравномерно: самый многочисленный класс: ядро, самый малочисленный: эндоплазматический ретикулум. Это необходимо учесть при последующем обучении и оценке моделей.